This notebook contains : tool calling, tool binding and tool excution 

In [1]:
%pip install -U langchain-groq langchain-core langchain-community requests pydantic



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
import os


In [3]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [4]:
print(multiply.invoke({'a':5, 'b':4}))

20


In [5]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given 2 numbers a and b this tool returns their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## tool binding

In [6]:
llm = ChatGroq(model="openai/gpt-oss-120b")


In [7]:
llm.invoke('hi')

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'We need to respond. The user says "hi". Simple greeting. We can respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 72, 'total_tokens': 110, 'completion_time': 0.078756354, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.002789426, 'prompt_tokens_details': None, 'queue_time': 0.379383562, 'total_time': 0.08154578}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12aab-19a5-77b2-9ec6-6f14b95b5530-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 38, 'total_tokens': 110, 'output_token_details': {'reasoning': 20}})

In [8]:
llm_with_tools = llm.bind_tools([multiply])

In [12]:
res = llm_with_tools.invoke('Hi how are you')

In [15]:
res

AIMessage(content="Hello! I'm doing great, thank you for asking. How can I assist you today?", additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. The user just says "Hi how are you". We should respond politely. No need for tools.'}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 133, 'total_tokens': 189, 'completion_time': 0.116472959, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.005642205, 'prompt_tokens_details': None, 'queue_time': 0.209642913, 'total_time': 0.122115164}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_69334c0be3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12aab-74eb-7f00-ab3c-a2d560d4cb3d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 133, 'output_tokens': 56, 'total_tokens': 189, 'output_token_details': {'reasoning': 29}})

In [24]:
res.tool_calls

[]

In [16]:
result = llm_with_tools.invoke('can you multiply 3 with 10')

In [22]:
result

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "can you multiply 3 with 10". We need to compute 3*10 = 30. Use the provided function multiply. Return result.', 'tool_calls': [{'id': 'fc_fddf3c3b-e709-468a-93f0-ea843d3fc58b', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 137, 'total_tokens': 206, 'completion_time': 0.145564327, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.005930253, 'prompt_tokens_details': None, 'queue_time': 0.305029395, 'total_time': 0.15149458}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c800245357', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12aab-f5e1-7332-b992-6e0c37c00ec9-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'fc_fddf3c3b-e709-468a-93f0-ea843d3fc58b', 'type': 't

In [17]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'fc_fddf3c3b-e709-468a-93f0-ea843d3fc58b',
 'type': 'tool_call'}

In [18]:
result.tool_calls[0]['args']

{'a': 3, 'b': 10}

In [19]:
multiply.invoke(result.tool_calls[0]['args'])

30

## tool execution

In [20]:
multiply.invoke(result.tool_calls[0])

ToolMessage(content='30', name='multiply', tool_call_id='fc_fddf3c3b-e709-468a-93f0-ea843d3fc58b')